# Edge Sampling Debug — `GO_1902781`

Runs the `sample_edges_for_nodes` logic inline (no function calls) for a single seed node  
to verify that self-loop exclusion and neighbor filtering work correctly.

In [4]:
import random
import sys
from pathlib import Path
import os

import polars as pl

PROJECT_ROOT = Path(os.getcwd())
sys.path.insert(0, str(PROJECT_ROOT))

from cli.commands.evals.utils import load_graph, load_node_metadata

## Config

In [5]:
NODES_PATH   = PROJECT_ROOT / "data/gold/kg/parquet/nodes.parquet"
EDGES_PATH   = PROJECT_ROOT / "data/gold/kg/parquet/edges.parquet"
GRAPH_MODE   = "undirected"
SEED         = 42

# The example node we are debugging
SEED_ID = "GO_1902781"

## Load graph + node metadata

In [6]:
random.seed(SEED)

G, node_types, edge_types = load_graph(NODES_PATH, EDGES_PATH, graph_mode=GRAPH_MODE)
node_metadata = load_node_metadata(NODES_PATH)

print(f"Graph: {G.number_of_nodes():,} nodes  {G.number_of_edges():,} edges")
print(f"Node types : {node_types}")
print(f"Edge types : {edge_types}")

[03/27/26 15:33:44] INFO     Loaded 192035 nodes with 10 node types: ['ANA', 'BPO', 'CCO', 'DIS',       ]8;id=234053;file:///Users/ayush/Documents/Zitnik_Lab/optimus/cli/commands/evals/utils.py\utils.py]8;;\:]8;id=146316;file:///Users/ayush/Documents/Zitnik_Lab/optimus/cli/commands/evals/utils.py#60\60]8;;\
                             'DRG', 'EXP', 'GEN', 'MFN', 'PHE', 'PWY']                                             

[03/27/26 15:35:21] INFO     Loaded 43631484 edges with 26 edge types: ['ANA-ANA', 'ANA-PRO',           ]8;id=571858;file:///Users/ayush/Documents/Zitnik_Lab/optimus/cli/commands/evals/utils.py\utils.py]8;;\:]8;id=91161;file:///Users/ayush/Documents/Zitnik_Lab/optimus/cli/commands/evals/utils.py#84\84]8;;\
                             'BPO-BPO', 'BPO-PRO', 'CCO-CCO', 'CCO-PRO', 'DIS-DIS', 'DIS-PHE',                     
                             'DIS-PRO', 'DRG-DIS', 'DRG-DRG', 'DRG-PHE', 'DRG-PRO', 'EXP-BPO',                     
                             'EXP-CCO', 'EXP-DIS', 'EXP-EXP', 'EXP-MFN', 'EXP-PRO', 'MFN-MFN',                     
                             'MFN-PRO', 'PHE-PHE', 'PHE-PRO', 'PRO-PRO', 'PWY-PRO', 'PWY-PWY']                     
                             (graph_mode=undirected)                                                               

[03/27/26 15:35:25] INFO     Loaded metadata for 192035 nodes                                          ]8;id=229258;file:///Users/ayush/Documents/Zitnik_Lab/optimus/cli/commands/evals/utils.py\utils.py]8;;\:]8;id=243962;file:///Users/ayush/Documents/Zitnik_Lab/optimus/cli/commands/evals/utils.py#122\122]8;;\

Graph: 192,035 nodes  43,631,484 edges
Node types : ['ANA', 'BPO', 'CCO', 'DIS', 'DRG', 'EXP', 'GEN', 'MFN', 'PHE', 'PWY']
Edge types : ['ANA-ANA', 'ANA-PRO', 'BPO-BPO', 'BPO-PRO', 'CCO-CCO', 'CCO-PRO', 'DIS-DIS', 'DIS-PHE', 'DIS-PRO', 'DRG-DIS', 'DRG-DRG', 'DRG-PHE', 'DRG-PRO', 'EXP-BPO', 'EXP-CCO', 'EXP-DIS', 'EXP-EXP', 'EXP-MFN', 'EXP-PRO', 'MFN-MFN', 'MFN-PRO', 'PHE-PHE', 'PHE-PRO', 'PRO-PRO', 'PWY-PRO', 'PWY-PWY']


## Build metadata lookup + nodes-with-names set

Mirrors the setup block inside `sample_edges_for_nodes`.

In [7]:
metadata_dict = {
    row["id"]: {"type": row["label"], "name": row["name"]}
    for row in node_metadata.iter_rows(named=True)
}

nodes_with_names = {
    node_id
    for node_id, meta in metadata_dict.items()
    if meta["name"] is not None and meta["name"] != ""
}

all_nodes = set(G.nodes())

print(f"Total nodes in graph      : {len(all_nodes):,}")
print(f"Nodes with valid names    : {len(nodes_with_names):,}")
print(f"Nodes WITHOUT valid names : {len(all_nodes - nodes_with_names):,}")

Total nodes in graph      : 192,035
Nodes with valid names    : 186,421
Nodes WITHOUT valid names : 5,614


## Inspect the seed node

In [8]:
seed_id = SEED_ID

print(f"seed_id : {seed_id}")
print(f"In graph: {seed_id in G}")
print(f"Has valid name: {seed_id in nodes_with_names}")

meta = metadata_dict.get(seed_id)
print(f"Metadata: {meta}")

seed_id : GO_1902781
In graph: True
Has valid name: True
Metadata: {'type': 'BPO', 'name': 'cellular response to nonane'}


In [ ]:
G.successors(seed_id)

## Compute neighbors  

Union of successors + predecessors (the same undirected neighbourhood used in `sample_edges_for_nodes`).

In [9]:
if seed_id in G:
    successors   = set(G.successors(seed_id))
    predecessors = set(G.predecessors(seed_id))
    neighbors    = successors | predecessors
else:
    successors = predecessors = neighbors = set()

print(f"Successors        : {len(successors):,}")
print(f"Predecessors      : {len(predecessors):,}")
print(f"Union (neighbors) : {len(neighbors):,}")
print()
print(f"Self-loop present (seed_id in neighbors)? {seed_id in neighbors}")

Successors        : 2
Predecessors      : 2
Union (neighbors) : 2

Self-loop present (seed_id in neighbors)? False


## Build `neighbors_list` — with the self-loop fix

In [12]:
# Before fix: no self-loop exclusion
neighbors_list_before = [n for n in neighbors if n in nodes_with_names]

# After fix: exclude self-loop
neighbors_list = [n for n in neighbors if n in nodes_with_names and n != seed_id]

removed_by_self_loop_fix = set(neighbors_list_before) - set(neighbors_list)

print(f"neighbors_list (before fix) : {len(neighbors_list_before):,}")
print(f"neighbors_list (after fix)  : {len(neighbors_list):,}")
print(f"Removed by self-loop fix    : {removed_by_self_loop_fix or 'none'}")

neighbors_list (before fix) : 2
neighbors_list (after fix)  : 2
Removed by self-loop fix    : none


## Browse the final `neighbors_list`

In [13]:
neighbor_rows = [
    {"node_id": n, **metadata_dict.get(n, {"type": None, "name": None})}
    for n in neighbors_list
]
neighbor_df = pl.DataFrame(neighbor_rows).sort("type", "node_id")
print(f"Total valid neighbors: {neighbor_df.height}")
neighbor_df

Total valid neighbors: 2


node_id,type,name
str,str,str
"""GO_1902779""","""BPO""","""cellular response to alkane"""
"""GO_1902780""","""BPO""","""response to nonane"""


In [14]:
# Count by neighbor type
neighbor_df.group_by("type").len().sort("len", descending=True)

type,len
str,u32
"""BPO""",2
